# Pilot sweep: one notebook, every result in one place

Runs short pilots of one arm and evaluates each on the test set. No project logic lives here:
the notebook clones the repository and calls `scripts/20_train.py` and `scripts/30_evaluate.py`.

**Setup:** Accelerator = **GPU T4 x2**, Internet = **on**. Then *Save Version* -> *Save & Run All*.

**What it does**
1. **Phase 1:** one pilot per learning rate in `LRS`, at the config's sampling temperature.
2. **Phase 2:** one pilot per sampling temperature in `TEMPS`, at the best phase-1 learning rate
   (or at `PHASE2_LR` if you set it).
3. Every pilot is evaluated on the test set. Two evaluations run at the same time, one per GPU.

Before any training, a **preflight** evaluates the untrained student on 8 questions on both GPUs,
through exactly the same path. If that fails, the notebook stops after a few minutes.

**Config:** each run's config = `BASE_CONFIG` + `COMMON` + the run's own change. Every config is
printed in full before training, with the changed lines marked, and saved next to its results.
Nothing in the repository is modified.

**Output** (everything under `/kaggle/working/`, kept by this one version):
- `pilots/<tag>/`: `pilot_config.yaml`, `train_stdout.log`, `train_log.jsonl`, `run_info.json`,
  `eval_stdout.log`, `predictions.jsonl`, `eval_summary.json`, `pilot_info.json`, `checkpoint/`
- `pilots/summary.csv`: one row per pilot (accuracy, loss, timings, ...), rewritten after each phase
- `pilots_small.zip`: everything above except the checkpoints

**Time** with the defaults (3 + 2 pilots of 100 steps, all 1,319 test questions): about 6 hours.
A time guard stops starting new work after `TIME_BUDGET_H`, below Kaggle's 12-hour limit.

In [ ]:
# ---- Settings: the only cell you edit ----
REPO = "https://github.com/redkitex/Knowledge-Distillation-and-Model-Compression.git"
REF = "main"                        # branch or commit
BASE_CONFIG = "configs/default.yaml"
ARM = "onpolicy"
SEED = 1
ACCOUNT = "T1"
MAX_STEPS = 100                     # pilot length; None = full run (500 steps)

# Applied to every run. Keys are "section.key" paths inside the config.
COMMON = {"eval.n_questions": None}     # None = all 1,319 test questions; e.g. 300 for a quick check

# Phase 1: one run per learning rate, at the config's sampling temperature (0.7 in default.yaml).
LRS = [5e-6, 1e-5, 2e-5]
# Phase 2: one run per sampling temperature, at the best phase-1 learning rate.
TEMPS = [0.5, 1.0]
PHASE2_LR = None                    # None = best phase-1 accuracy; or force one, e.g. 1e-5

REF_STUDENT0_ACC = 0.6202           # untrained student on all 1,319 questions, for the summary
TIME_BUDGET_H = 11.0                # no new training or evaluation is started after this

In [ ]:
# Private repository? Add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets) and uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")

# Optional, for faster Hugging Face downloads: a secret named HF_TOKEN.
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print("GPUs:", n_gpus)
assert n_gpus >= 2, "Needs Accelerator = GPU T4 x2: teacher on GPU 1, student on GPU 0, evaluations on both"

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q {REPO} repo
%cd /kaggle/working/repo
!git checkout -q {REF} && git log -1 --oneline
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# ---- Helpers: no need to edit ----
import copy
import csv
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import yaml

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "repo"
OUT = WORK / "pilots"                # everything in here is kept in this version's Output
SCRATCH = Path("/tmp/kd_eval")      # throwaway folders for the evaluations
EVAL_RUN = f"{ARM}_s{SEED}"         # standard run name; scripts/30_evaluate.py parses it reliably
POLL_S, HEARTBEAT_S = 10, 120
TRAIN_EST_H, EVAL_EST_H = 0.8, 0.9  # rough upper estimates, used only by the time guard

T0 = time.time()
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "_configs").mkdir(exist_ok=True)
ENV = dict(os.environ, PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
BASE = yaml.safe_load(open(REPO_DIR / BASE_CONFIG, encoding="utf-8"))
RUNS = []                           # one dict per run, in the order they were planned


def hours():
    return (time.time() - T0) / 3600


def _node(cfg, path):
    """Return (parent dict, key) for a "section.key" path; fail loudly on a typo."""
    *parents, leaf = path.split(".")
    node = cfg
    for key in parents:
        if not isinstance(node, dict) or key not in node:
            raise KeyError(f"{BASE_CONFIG} has no section for {path!r}")
        node = node[key]
    if not isinstance(node, dict) or leaf not in node:
        raise KeyError(f"{BASE_CONFIG} has no key {path!r}")
    return node, leaf


def get(cfg, path):
    node, leaf = _node(cfg, path)
    return node[leaf]


def flatten(cfg, prefix=""):
    flat = {}
    for key, value in cfg.items():
        if isinstance(value, dict):
            flat.update(flatten(value, f"{prefix}{key}."))
        else:
            flat[f"{prefix}{key}"] = value
    return flat


def build_cfg(overrides):
    cfg = copy.deepcopy(BASE)
    for path, value in {**COMMON, **overrides}.items():
        node, leaf = _node(cfg, path)
        node[leaf] = value
    return cfg


def make_run(phase, overrides):
    cfg = build_cfg(overrides)
    lr = float(get(cfg, "train.lr"))
    temp = float(get(cfg, "generation.sampling.temperature"))
    tag = f"pilot_{ARM}_lr{lr:g}_t{temp:g}_s{SEED}"
    path = OUT / "_configs" / f"{tag}.yaml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    base_flat = flatten(BASE)
    changed = {k: base_flat.get(k) for k, v in flatten(cfg).items() if base_flat.get(k) != v}
    return {"tag": tag, "phase": phase, "lr": lr, "temperature": temp, "overrides": overrides,
            "config": path, "changed": changed, "trained": False, "evaluated": False,
            "status": "planned"}


def show(run):
    print(f"\n=== {run['tag']}  (phase {run['phase']}) ===")
    for key, value in flatten(yaml.safe_load(open(run["config"], encoding="utf-8"))).items():
        mark = f"    <-- changed, base: {run['changed'][key]!r}" if key in run["changed"] else ""
        print(f"  {key} = {value!r}{mark}")


def read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def last_line(path):
    try:
        text = Path(path).read_text(encoding="utf-8", errors="replace")
    except FileNotFoundError:
        return ""
    lines = [line.split("\r")[-1].strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return lines[-1][:160] if lines else ""


def stream(cmd, cwd, log_path, env):
    """Run a command, print its output live and keep a copy in log_path."""
    with open(log_path, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return proc.wait()


def train(run):
    tag, dst = run["tag"], OUT / run["tag"]
    if hours() + TRAIN_EST_H + EVAL_EST_H > TIME_BUDGET_H:
        run["status"] = "skipped: time budget"
        print(f"SKIP {tag}: {hours():.1f} h used, not enough time left")
        return
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(run["config"], dst / "pilot_config.yaml")
    cmd = [sys.executable, "scripts/20_train.py", "--config", str(run["config"]),
           "--arm", ARM, "--seed", str(SEED), "--run-id", tag, "--account", ACCOUNT]
    if MAX_STEPS:
        cmd += ["--max-steps", str(MAX_STEPS)]
    print(f"\n[{hours():.2f} h] TRAIN {tag}", flush=True)
    rc = stream(cmd, REPO_DIR, dst / "train_stdout.log", ENV)
    src = REPO_DIR / "results" / "runs" / tag
    if src.exists():                 # move the run out of the repo, next to its config and logs
        for item in src.iterdir():
            shutil.move(str(item), str(dst / item.name))
        shutil.rmtree(src, ignore_errors=True)
    run["trained"] = rc == 0 and (dst / "checkpoint").is_dir()
    run["status"] = "trained" if run["trained"] else f"train failed, exit code {rc}"
    print(f"[{hours():.2f} h] {tag}: {run['status']}", flush=True)


def evaluate(runs):
    """Evaluate trained runs two at a time, one per GPU.

    Each evaluation runs in its own scratch folder where the checkpoint is linked as
    results/runs/<ARM>_s<SEED>/checkpoint, the standard name scripts/30_evaluate.py expects.
    Its predictions and summary are then copied next to the run's other files.
    """
    todo = [r for r in runs if r["trained"]]
    for i in range(0, len(todo), 2):
        pair = todo[i:i + 2]
        if hours() + EVAL_EST_H > TIME_BUDGET_H:
            for run in pair:
                run["status"] = "trained, evaluation skipped: time budget"
            print(f"SKIP evaluation of {[r['tag'] for r in pair]}: time budget")
            continue
        jobs = []
        for gpu, run in enumerate(pair):
            work = SCRATCH / run["tag"]
            shutil.rmtree(work, ignore_errors=True)
            run_dir = work / "results" / "runs" / EVAL_RUN
            run_dir.mkdir(parents=True)
            (run_dir / "checkpoint").symlink_to(OUT / run["tag"] / "checkpoint")
            log_path = OUT / run["tag"] / "eval_stdout.log"
            log = open(log_path, "w", encoding="utf-8")
            cmd = [sys.executable, str(REPO_DIR / "scripts" / "30_evaluate.py"),
                   "--config", str(run["config"]), "--run", EVAL_RUN]
            proc = subprocess.Popen(cmd, cwd=work, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)),
                                    stdout=log, stderr=subprocess.STDOUT, text=True)
            jobs.append((run, proc, log, log_path, run_dir, gpu))
            print(f"[{hours():.2f} h] EVAL {run['tag']} on GPU {gpu}", flush=True)
        last = time.time()
        while any(job[1].poll() is None for job in jobs):
            time.sleep(POLL_S)
            if time.time() - last >= HEARTBEAT_S:
                last = time.time()
                for run, proc, _, log_path, _, gpu in jobs:
                    state = "running" if proc.poll() is None else "finished"
                    print(f"[{hours():.2f} h] GPU {gpu} {run['tag']}: {state} | {last_line(log_path)}",
                          flush=True)
        for run, proc, log, log_path, run_dir, gpu in jobs:
            log.close()
            for name in ("predictions.jsonl", "eval_summary.json"):
                if (run_dir / name).exists():
                    shutil.copy(run_dir / name, OUT / run["tag"] / name)
            summary = read_json(OUT / run["tag"] / "eval_summary.json")
            run["evaluated"] = proc.returncode == 0 and "accuracy" in summary
            if run["evaluated"]:
                run["status"] = "done"
                print(f"[{hours():.2f} h] {run['tag']}: accuracy {summary['accuracy']:.2%}", flush=True)
            else:
                run["status"] = f"evaluation failed, exit code {proc.returncode}"
                print(f"[{hours():.2f} h] {run['tag']}: {run['status']}. Last lines of the log:")
                print("".join(Path(log_path).read_text(errors="replace").splitlines(True)[-15:]))


def write_summary():
    rows = []
    for run in RUNS:
        folder = OUT / run["tag"]
        info = read_json(folder / "run_info.json")
        ev = read_json(folder / "eval_summary.json")
        s = info.get("summary", {})
        acc = ev.get("accuracy")
        rows.append({
            "tag": run["tag"], "phase": run["phase"], "lr": run["lr"],
            "sampling_temperature": run["temperature"], "status": run["status"],
            "accuracy": acc,
            "vs_ref_student0_points": None if acc is None else round(100 * (acc - REF_STUDENT0_ACC), 2),
            "correct": ev.get("correct"), "n_questions": ev.get("n"),
            "eval_truncated_fraction": ev.get("truncated_fraction"),
            "eval_mean_response_tokens": ev.get("mean_response_tokens"),
            "final_loss_mean_last_10": s.get("final_loss_mean_last_10"),
            "train_unfinished_sequences": s.get("unfinished_sequences"),
            "oom_retries": s.get("oom_retries"),
            "train_seconds": s.get("train_seconds"), "eval_seconds": ev.get("elapsed_seconds"),
        })
        if folder.exists():
            with open(folder / "pilot_info.json", "w", encoding="utf-8") as f:
                json.dump({"tag": run["tag"], "phase": run["phase"], "status": run["status"],
                           "overrides": run["overrides"], "common": COMMON,
                           "changed_vs_base": run["changed"], "base_config": BASE_CONFIG,
                           "eval_run_name": EVAL_RUN,
                           "note": "predictions.jsonl carries run_id/arm of EVAL_RUN; "
                                   "this folder's tag is the real identity of the run"},
                          f, indent=2, default=str)
    if rows:
        with open(OUT / "summary.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        try:
            import pandas as pd
            print(pd.DataFrame(rows)[["tag", "status", "accuracy", "vs_ref_student0_points",
                                      "final_loss_mean_last_10", "train_seconds"]].to_string(index=False))
        except ImportError:
            for row in rows:
                print(row)

In [ ]:
# ---- Plan: every config is built and checked here, so a typo stops the notebook in seconds ----
for temp in TEMPS:      # phase-2 keys are checked now; the real phase-2 configs are built later
    build_cfg({"train.lr": LRS[0], "generation.sampling.temperature": temp})
_arm, _seed = EVAL_RUN.rsplit("_s", 1)  # the same split scripts/30_evaluate.py does
int(_seed)

PHASE1 = [make_run(1, {"train.lr": lr}) for lr in LRS]
tags = [run["tag"] for run in PHASE1]
assert len(set(tags)) == len(tags), f"two phase-1 runs would get the same name: {tags}"
RUNS.extend(PHASE1)

print("Base config:", BASE_CONFIG, "| common changes:", COMMON)
for run in PHASE1:
    show(run)
print(f"\nPlan: {len(PHASE1)} phase-1 pilots, then {len(TEMPS)} phase-2 pilots. "
      f"Expected about {0.2 + len(PHASE1) * 0.7 + 1.6 + len(TEMPS) * 0.75 + 0.8:.1f} h.")

In [ ]:
# ---- Preflight: an 8-question evaluation on both GPUs through the exact same path ----
# Catches path or naming problems in a few minutes, before hours of training.
import gc
import torch
from transformers import AutoModelForCausalLM

pre_a, pre_b = OUT / "_preflight_gpu0", OUT / "_preflight_gpu1"
pre_a.mkdir(exist_ok=True)
pre_b.mkdir(exist_ok=True)
if not (pre_a / "checkpoint").exists():   # the untrained student, saved like a trained checkpoint
    model = AutoModelForCausalLM.from_pretrained(BASE["student"], torch_dtype=torch.float16)
    model.save_pretrained(pre_a / "checkpoint")
    del model
    gc.collect()
if not (pre_b / "checkpoint").exists():
    (pre_b / "checkpoint").symlink_to(pre_a / "checkpoint")
pre_cfg = OUT / "_configs" / "_preflight.yaml"
with open(pre_cfg, "w", encoding="utf-8") as f:
    yaml.safe_dump(build_cfg({"eval.n_questions": 8}), f, sort_keys=False)
PRE = [{"tag": p.name, "config": pre_cfg, "trained": True, "evaluated": False, "status": ""}
       for p in (pre_a, pre_b)]
evaluate(PRE)
assert all(r["evaluated"] for r in PRE), "Preflight evaluation failed (log above). Nothing was trained."
shutil.rmtree(pre_b)
shutil.rmtree(pre_a)
pre_cfg.unlink()
print(f"Preflight OK after {hours():.2f} h")

In [ ]:
# ---- Phase 1: learning rates ----
try:
    for run in PHASE1:
        train(run)
    evaluate(PHASE1)
except Exception:
    traceback.print_exc()
write_summary()     # written after every phase, so a later problem loses nothing that finished

In [ ]:
# ---- Phase 2: sampling temperatures at the best phase-1 learning rate ----
PHASE2 = []
try:
    acc1 = {}
    for run in PHASE1:
        acc = read_json(OUT / run["tag"] / "eval_summary.json").get("accuracy")
        if acc is not None:
            acc1[run["lr"]] = acc
            print(f"phase 1: lr {run['lr']:g} -> {acc:.2%} "
                  f"({100 * (acc - REF_STUDENT0_ACC):+.1f} points vs ref_student0)")
    best_lr = PHASE2_LR if PHASE2_LR is not None else (max(acc1, key=acc1.get) if acc1 else None)
    if best_lr is None:
        print("Phase 2 skipped: no phase-1 accuracy available and PHASE2_LR is None.")
    else:
        how = "forced by PHASE2_LR" if PHASE2_LR is not None else "best phase-1 accuracy"
        print(f"Phase 2 learning rate: {best_lr:g} ({how})")
        known = {run["tag"] for run in RUNS}
        for temp in TEMPS:
            run = make_run(2, {"train.lr": best_lr, "generation.sampling.temperature": temp})
            if run["tag"] in known:
                print("Already run in phase 1, skipped:", run["tag"])
                continue
            PHASE2.append(run)
            RUNS.append(run)
            show(run)
        for run in PHASE2:
            train(run)
        evaluate(PHASE2)
except Exception:
    traceback.print_exc()
write_summary()

In [ ]:
# ---- Package: one zip with everything except checkpoints; checkpoints stay in pilots/<tag>/ ----
write_summary()
small = Path("/tmp/pilots_small")
shutil.rmtree(small, ignore_errors=True)
shutil.copytree(OUT, small, ignore=shutil.ignore_patterns("checkpoint"))
shutil.make_archive(str(WORK / "pilots_small"), "zip", small)
os.chdir(WORK)
shutil.rmtree(REPO_DIR, ignore_errors=True)   # tidy Output: only pilots/ and pilots_small.zip remain
print(f"\nFinished after {hours():.2f} h. Download pilots_small.zip from the Output tab.")
!du -sh {OUT}/*/checkpoint 2>/dev/null; ls -la {WORK}